# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The dataset provides two non-overlapping windows — `prev_30d` (older) and `last_30d` (more recent) — which makes it possible to define a label from an outcome period and restrict features to what was known *before* that period, rather than mixing the two. Label: `needs_refresh = 1` if `trend_direction == "down"`. Split: **client-holdout**, not random rows — client page counts range from 3 to 7,008, and a random split would let one client's pattern leak across train and test.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

# Label from the OUTCOME window (last_30d vs prev_30d)
df["needs_refresh"] = (df["trend_direction"] == "down").astype(int)

# Feature set restricted to the PAST window only - excludes anything that
# overlaps with or defines the outcome window (see Section 3 for why)
safe_features = [
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "word_count", "char_count",
    "content_type", "main_intent",
    "search_volume", "competition", "cpc",
    "client_id",  # needed for the split, not a model feature
]

df["ctr_prev_30d"] = df["clicks_prev_30d"] / df["impressions_prev_30d"].replace(0, np.nan)
safe_features.append("ctr_prev_30d")

rng = np.random.RandomState(42)
clients = df["client_id"].unique()
test_clients = rng.choice(clients, size=max(1, int(len(clients) * 0.25)), replace=False)

train_df = df[~df["client_id"].isin(test_clients)].copy()
test_df  = df[df["client_id"].isin(test_clients)].copy()

print(f"Train: {train_df.shape[0]} rows, {train_df['client_id'].nunique()} clients")
print(f"Test:  {test_df.shape[0]} rows, {test_df['client_id'].nunique()} clients")
print(f"\nLabel balance - train: {train_df['needs_refresh'].mean():.2%}, test: {test_df['needs_refresh'].mean():.2%}")

# --- Missing-data handling ---
# Numeric gaps: filled with the TRAIN median only (never test - avoids leaking test
# statistics into the fill values), plus a _was_missing flag per column, since absence
# of data can itself be informative. Categorical gaps: an explicit "unknown" category
# rather than dropped rows - dropping would bias toward whichever content is well-tracked.
numeric_cols = ["word_count", "char_count", "search_volume", "competition", "cpc", "ctr_prev_30d"]
categorical_cols = ["content_type", "main_intent"]

for col in numeric_cols:
    median_value = train_df[col].median()  # TRAIN ONLY
    train_df[f"{col}_was_missing"] = train_df[col].isna().astype(int)
    test_df[f"{col}_was_missing"] = test_df[col].isna().astype(int)
    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)  # same train median applied to test

for col in categorical_cols:
    train_df[col] = train_df[col].fillna("unknown")
    test_df[col] = test_df[col].fillna("unknown")

final_features = (
    numeric_cols
    + [f"{c}_was_missing" for c in numeric_cols]
    + categorical_cols
    + ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
       "content_age_days", "age_tier_order", "days_since_last_update"]
)

print("\nFinal feature count:", len(final_features))
print("Remaining missing values, train:", train_df[final_features].isna().sum().sum())
print("Remaining missing values, test: ", test_df[final_features].isna().sum().sum())

Train: 26494 rows, 24 clients
Test:  3506 rows, 8 clients

Label balance - train: 54.45%, test: 52.37%

Final feature count: 20
Remaining missing values, train: 0
Remaining missing values, test:  0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing? | Type | Available before prediction? |
|---|---|---|---|---|
| `impressions_prev_30d` | Search impressions in the older 30-day window | None observed | Numeric | Yes — fully past-window |
| `clicks_prev_30d` | Clicks in the older 30-day window | None observed | Numeric | Yes |
| `sessions_prev_30d` | Analytics sessions in the older 30-day window | None observed | Numeric | Yes |
| `ctr_prev_30d` | `clicks_prev_30d / impressions_prev_30d`, engineered here | Inherits 0-impression rows as NaN, train-median filled + flagged | Numeric (engineered) | Yes — built only from past-window columns |
| `content_age_days` | Days since the page was published, as of the export | None observed | Numeric | Yes — a static page attribute, not window-dependent |
| `age_tier_order` | Ordinal bucket of content age | None observed | Numeric (ordinal) | Yes |
| `days_since_last_update` | Days since the page was last edited | None observed | Numeric | Yes |
| `word_count` / `char_count` | Content length metadata | ~26% missing — train-median filled + `_was_missing` flag | Numeric | Yes — static content attribute |
| `search_volume` / `competition` / `cpc` | Keyword-level SEO metadata | ~8% missing — train-median filled + `_was_missing` flag | Numeric | Yes — keyword-level, not outcome-window dependent |
| `content_type` | Page category (blog, landing, product, etc.) | ~8% missing — filled as `"unknown"` | Categorical | Yes |
| `main_intent` | Primary search intent for the page's target keyword | ~8% missing — filled as `"unknown"` | Categorical | Yes |
| `client_id` | Hashed client identifier, used only for the client-holdout split | None observed | ID (not a model feature) | N/A — excluded from `final_features` |

**Not in this table on purpose:** `ctr`, `avg_position`, every `*_90d` column, and `trend_pct` — all excluded for leakage reasons detailed in Section 3.

In [3]:
missing_pct = df[["word_count", "char_count", "search_volume", "competition", "cpc", "main_intent"]].isna().mean() * 100
print("Missingness by column (full dataset, before any split):")
print(missing_pct.round(1))

Missingness by column (full dataset, before any split):
word_count       25.7
char_count       25.7
search_volume     8.2
competition       8.2
cpc               8.2
main_intent       7.9
dtype: float64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Two direct checks below, not just a claim. Both confirm real leakage: `trend_pct` correlates
**1.0000** with the manual percentage change in impressions across the outcome window, and `ctr`
correlates **1.0000** with the 90-day click/impression ratio — both are direct transforms of the
outcome, not independent predictors of it.

In [7]:
# Check 1: trend_pct vs the manual outcome-window PERCENTAGE change it should be derived
# from. "trend_pct" implies a relative change, not an absolute one - correcting that here.
manual_pct_change = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"])
    / df["impressions_prev_30d"].replace(0, np.nan)
) * 100
corr_trend_pct = df["trend_pct"].corr(manual_pct_change)
print(f"trend_pct vs pct change in impressions (last_30d vs prev_30d): corr = {corr_trend_pct:.4f}")
print("  -> near 1.0 means trend_pct is a direct transform of the outcome window, not an independent signal.\n")

# Check 2: ctr vs the 90-day aggregate ratio it's built from.
manual_ctr_90d = df["clicks_90d"] / df["impressions_90d"].replace(0, np.nan)
corr_ctr = df["ctr"].corr(manual_ctr_90d)
print(f"ctr vs (clicks_90d / impressions_90d): corr = {corr_ctr:.4f}")
print("  -> near 1.0 confirms 'ctr' is the 90-day ratio, and the 90-day window mathematically")
print("     contains last_30d - so any *_90d column carries outcome-window information too.\n")

# Check 3: avg_position - is it measured over a window that includes the outcome period?
# No manual reconstruction available (single current value, not two windows to compare),
# which is itself the red flag: there's no way to confirm it reflects only the PRE-outcome
# state, so it's treated as unsafe rather than assumed innocent.
print("avg_position: single current-state column, no prev/last window split available -")
print("  cannot confirm it excludes the outcome period, so it is excluded on that basis alone.")

trend_pct vs pct change in impressions (last_30d vs prev_30d): corr = 1.0000
  -> near 1.0 means trend_pct is a direct transform of the outcome window, not an independent signal.

ctr vs (clicks_90d / impressions_90d): corr = 1.0000
  -> near 1.0 confirms 'ctr' is the 90-day ratio, and the 90-day window mathematically
     contains last_30d - so any *_90d column carries outcome-window information too.

avg_position: single current-state column, no prev/last window split available -
  cannot confirm it excludes the outcome period, so it is excluded on that basis alone.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`trend_pct`** — correlates ~1.0 with the manual `impressions_last_30d - impressions_prev_30d` calculation (see Section 3, Check 1). It's a restatement of the outcome, not a predictor of it.
- **`ctr`** — correlates ~1.0 with `clicks_90d / impressions_90d` (Section 3, Check 2). Since the label is defined from the `last_30d` window and `90d` mathematically contains `last_30d`, this column leaks outcome-period information.
- **Every `*_90d` column** (`impressions_90d`, `clicks_90d`, `sessions_90d`, etc.) — same reasoning as `ctr`: the 90-day window overlaps the outcome window by construction, not by coincidence.
- **`avg_position`** — no prev/last window split exists for this column to confirm it's pre-outcome only (Section 3, Check 3). Absence of evidence that it's safe was treated as evidence it isn't, rather than the other way around.
- **`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`** — these define the outcome window directly; using them as features would mean predicting the label from the label's own inputs.

**Everything kept** in `final_features` (Section 1) comes from `prev_30d`, static content/keyword metadata, or engineered columns built exclusively from `prev_30d` fields — nothing here shares a window with the label.

In [8]:
excluded_cols = [
    "trend_pct", "ctr", "avg_position",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
]
still_present = [c for c in excluded_cols if c in final_features]
print("Leakage check - excluded columns accidentally left in final_features:", still_present or "None. Clean.")

Leakage check - excluded columns accidentally left in final_features: None. Clean.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.